# 01 · From a language model to an application

**Session 01 · Building AI applications for research and data · notebook 1 of 6**

## Objective

See what a model does with a research question on its own, what changes when it is given instructions and context, and why instructions and context are the first two components of an application.

## The application after this notebook

```
question ──┐
            ├──> [ instructions + context + question ] ──> model ──> answer
context ───┘
```

The worked example for the whole session is one question:

> How have internet access and economic development evolved in Rwanda and Kenya? Compare the recent available data, summarize the main patterns, and cite the sources.

## How to run it

Run the cells in order. Without an API key the model outputs are recorded ones, labelled `[recorded output: ...]`. With a key in `WORKSHOP_API_KEY` the same cells call the model live.

In [1]:
# Setup. In Colab this clones the repository and installs the dependencies; locally it uses the folder as is.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/avsolatorio/ai4data.git"   # replace with the published repository when released
BRANCH = "feat/training-series"
SESSION = "training/session-01-building-ai-applications/notebooks"

if not pathlib.Path("workshop.py").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/ai4data"], check=True)
    os.chdir(f"/content/ai4data/{SESSION}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "pydantic", "scikit-learn", "requests"], check=True)
sys.path.insert(0, os.getcwd())

# Optional: a key for an OpenAI-compatible API. Without one, the notebook uses recorded outputs and says so.
try:
    from google.colab import userdata  # type: ignore
    for name in ("WORKSHOP_API_KEY", "WORKSHOP_BASE_URL", "WORKSHOP_MODEL"):
        if userdata.get(name):
            os.environ[name] = userdata.get(name)
except Exception:
    pass

import workshop as w
llm = w.LLM()
print("Model client:", llm.mode())

Model client: recorded outputs (no WORKSHOP_API_KEY set)


## 1. The function the whole session builds on

Three parts go into every call: **instructions** (the system message), **context** (evidence supplied to the model), and the **question**. Keeping them separate is the first design decision of an application, because each is changed by a different person for a different reason.

In [2]:
INSTRUCTIONS_BASIC = "You are a research assistant. Answer the question."

INSTRUCTIONS_GROUNDED = """You are a research assistant for development data.
Answer only from the supplied context. Cite the document id in square brackets after each figure.
If the context does not contain what is needed, say so and give the closest available information."""

INSTRUCTIONS_CLARIFY = INSTRUCTIONS_GROUNDED + """
If the question is ambiguous, say which readings are possible and ask which one is meant before answering."""


def ask(question, instructions=INSTRUCTIONS_BASIC, context=None, scenario="unnamed"):
    """Build the messages from the three parts and call the model."""
    user = question if context is None else f"CONTEXT:\n{context}\n\nQUESTION: {question}"
    messages = [{"role": "system", "content": instructions}, {"role": "user", "content": user}]
    return llm.complete(messages, scenario=scenario)


print(w.MAIN_QUESTION)

How have internet access and economic development evolved in Rwanda and Kenya? Compare the recent available data, summarize the main patterns, and cite the sources.


## 2. Zero context: the model answers from its training data

The baseline. No evidence is supplied, so the model answers from what it absorbed in training.

In [3]:
answer = ask(w.MAIN_QUESTION, scenario="nb01_zero_context")
w.show(answer)

[recorded output: nb01_zero_context]
Based on what I know up to my training data, both countries have expanded internet access and grown
their economies, though from different starting points.

Kenya is one of East Africa's most connected economies. Internet penetration is often reported above
80 percent of the population when measured by subscriptions, and mobile money through M-Pesa is
widespread. GDP per capita is roughly 2,000 US dollars, and growth has averaged around 5 percent a
year over the past decade.

Rwanda has invested heavily in broadband infrastructure and e-government. Internet penetration is
commonly cited at around 30 percent of the population, and GDP per capita is close to 1,000 US
dollars, with growth frequently above 7 percent.

Please note that these figures are approximate and may be out of date. For precise, current values
you should consult the World Bank's World Development Indicators or the ITU.


Read the answer against the data. The cell below prints the latest values from the World Development Indicators that the session uses as its reference.

In [4]:
for ind in ["IT.NET.USER.ZS", "NY.GDP.PCAP.CD"]:
    for country in ["Kenya", "Rwanda"]:
        r = w.get_indicator(ind, country, start=2023)
        print(f"{r['name']:50s} {country:7s}", [(o["year"], round(o["value"], 1)) for o in r["observations"]])

Individuals using the Internet (% of population)   Kenya   [(2023, 32.1), (2024, 35.0)]
Individuals using the Internet (% of population)   Rwanda  [(2023, 28.8), (2024, 31.7)]
GDP per capita (current US$)                       Kenya   [(2023, 1942.6), (2024, 2133.5)]
GDP per capita (current US$)                       Rwanda  [(2023, 1061.1), (2024, 1059.9)]


What to notice:

- The figures are approximate, undated, and in places wrong. "Internet penetration above 80 percent" is a subscriptions figure presented as internet use; the indicator for individuals using the internet stood at 35 percent in Kenya in 2024.
- There are no sources a reader could check.
- The model hedges, which is the right behaviour for a model with no evidence, and of no use to a reader who needs a number.

Prompting changes the style of this answer. It cannot give the model data it does not have.

## 3. The same question with context

The four country briefs from the session's document collection are supplied as context. The instructions now require citations and allow the model to say when something is missing.

In [5]:
docs = {d.id: d for d in w.load_documents()}
context = "\n\n".join(f"[{d.id}]\n{d.text}" for d in [docs["kenya-digital"], docs["rwanda-digital"], docs["kenya-economy"], docs["rwanda-economy"]])
print(f"{len(context):,} characters of context, roughly {len(context)//4:,} tokens\n")

answer = ask(w.MAIN_QUESTION, INSTRUCTIONS_GROUNDED, context, scenario="nb01_with_context")
w.show(answer)

4,076 characters of context, roughly 1,019 tokens

[recorded output: nb01_with_context]
Internet use. In Kenya, the share of individuals using the internet rose from 7.2 percent of the
population in 2010 to 35.0 percent in 2024 [kenya-digital]. In Rwanda it rose from 8.0 percent in
2010 to 31.7 percent in 2024 [rwanda-digital]. Kenya's fastest growth came between 2010 and 2015;
Rwanda's came between 2015 and 2020. Both countries ended the period with about a third of the
population online.

Economic development. Kenya's GDP per capita in current US dollars rose from 1,092 in 2010 to 2,133
in 2024, with a dip to 1,943 in 2023 that coincided with a currency depreciation [kenya-economy].
Rwanda's rose from 587 in 2010 to 1,060 in 2024, passing 1,000 in 2022 [rwanda-economy]. Rwanda grew
faster in most years, above 7 percent except in 2020, while Kenya grew in every year except 2020.

Patterns. Kenya has the higher level on both indicators; Rwanda has the faster growth on both.
Internet us

The figures now come from the briefs, each one is cited, and the comparison is specific. The model did none of the finding; the application did, by putting the right text in front of it.

## 4. Handling missing information

An application has to behave well when the evidence does not contain the answer. The instruction "say so and give the closest available information" is what makes the difference.

In [6]:
answer = ask("What share of Kenya's population used the internet in 2025?", INSTRUCTIONS_GROUNDED, docs["kenya-digital"].text, scenario="nb01_missing_info")
w.show(answer)

[recorded output: nb01_missing_info]
The supplied material contains no observation for 2025. The latest available year for the share of
Kenya's population using the internet is 2024, at 35.0 percent [kenya-digital]. If a 2025 value is
needed, it will have to come from a later release of the World Development Indicators or from the
ITU.


## 5. An ambiguous question, with and without an instruction

"How connected is Kenya?" can mean internet use, mobile subscriptions, broadband, or something else. Compare what the model does with the basic instructions and with an instruction to ask.

In [7]:
print("Without an instruction to clarify:\n")
w.show(ask("How connected is Kenya?", INSTRUCTIONS_BASIC, docs["kenya-digital"].text, scenario="nb01_ambiguous_no_instruction"))
print("\n" + "=" * 100 + "\nWith the instruction:\n")
w.show(ask("How connected is Kenya?", INSTRUCTIONS_CLARIFY, docs["kenya-digital"].text, scenario="nb01_ambiguous_with_instruction"))

Without an instruction to clarify:

[recorded output: nb01_ambiguous_no_instruction]
Kenya is highly connected. Mobile cellular subscriptions reached 126.5 per 100 people in 2024, which
is more than one subscription per person, and 35.0 percent of the population used the internet in
the same year. Kenya is also the home of M-Pesa, so mobile money use is very high. Overall, Kenya is
one of the most connected countries in East Africa.

With the instruction:

[recorded output: nb01_ambiguous_with_instruction]
"Connected" can mean several things, and the supplied material measures two of them. Which do you
mean?

- Share of individuals using the internet: 35.0 percent of the population in 2024 [kenya-digital].
- Mobile cellular subscriptions: 126.5 per 100 people in 2024, a count of subscriptions that can
exceed the number of people [kenya-digital].

The two measures answer different questions. Subscriptions passed one per person around 2020, while
internet use stood at about a third of th

The first answer picks a reading and runs with it. The second names the readings and asks. For a research application the second is the behaviour to specify, because a confident answer to the wrong question looks the same as a confident answer to the right one.

## 6. What the model received

The last call's messages, as sent. The three parts are visible: system instructions, then the context and the question in the user message.

In [8]:
for m in llm.calls[-1]["messages"]:
    print(f"--- {m['role']} ({len(m['content']):,} characters)")
    print(m["content"][:400] + (" …" if len(m["content"]) > 400 else ""))
    print()

--- system (354 characters)
You are a research assistant for development data.
Answer only from the supplied context. Cite the document id in square brackets after each figure.
If the context does not contain what is needed, say so and give the closest available information.
If the question is ambiguous, say which readings are possible and ask which one is meant before answering.

--- user (1,345 characters)
CONTEXT:
# Kenya: internet use and mobile subscriptions, 2010 to 2024

## Internet use

The share of individuals using the internet in Kenya rose from 7.2 percent of the population in 2010 to 16.6 percent in 2015 and 24.7 percent in 2020. Growth continued after 2020: 29.4 percent in 2022, 32.1 percent in 2023, and 35.0 percent in 2024, the latest year available.

Over the full period the share inc …



## 7. What context costs

Four short briefs took about a thousand tokens. A real collection has thousands of documents and cannot be pasted into every call: the context window has a limit, cost grows with every token, and the relevant passages are a small part of the whole. The next two notebooks prepare a collection so that the application can select what to supply.

## Key points

- An application separates instructions, context, and the question. Each is changed by a different person for a different reason.
- Instructions shape behaviour: citations, what to do when information is missing, when to ask. They cannot supply data the model does not have.
- Supplied context is what makes the answer specific and checkable. Selecting it is the job of the rest of the application.

## Exercise

Change `INSTRUCTIONS_GROUNDED` so that the answer is a table with one row per country and columns for the indicator, the latest year, the value, and the source. Rerun section 3. With a live key the output changes; in recorded mode the cell returns the recorded answer, so note what you would expect instead.

## Where this connects to the program

- [Practical Guide to Serving Official Statistics to AI Agents](https://worldbank.github.io/ai4data/cookbook/serving-statistics-to-agents/): the design of tools over a statistical catalog, in full.
- [Practical Guide to Evaluation Suites for Statistical AI](https://worldbank.github.io/ai4data/cookbook/evaluation-suites/): how to build and run the tests.
- [Proof-Carrying Numbers](https://worldbank.github.io/ai4data/docs/pcn/): the claim-tag protocol the verifier in this session follows.
- [Data360 MCP server](https://github.com/worldbank/data360-mcp): the tools of notebook 04, exposed to any MCP client over the World Bank's data platform.
- [End-to-end reference implementation](https://worldbank.github.io/ai4data/docs/reference-implementation/): documentation, validation, publication, agent access, and verification in one run.